<a id="curation-lab"></a>
# VideoDB Curation: QA a Dataset from the Curation Lab

The **Curation Lab** applies known problems to a LeRobot v3 dataset in Azure Blob, in place. This
notebook connects that dataset to Curation, then runs **validate** and **audit** and shows what
Curation returns.

Connect once. After each scenario you apply in the Lab, re-run **Validate** and **Audit**.

<a href="https://colab.research.google.com/github/video-db/videodb-cookbook/blob/main/guides/curation/curation_lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a id="setup"></a>
## 1. Set up

Set `VIDEO_DB_BASE_URL` to a deployment with Curation enabled, and enter your API key when prompted.
The SDK does not wrap the dataset endpoints yet, so the cells call them with `conn.get` and `conn.post`.

In [ ]:
!pip install -q "videodb>=0.5.0" python-dotenv pandas

In [ ]:
import json
import os
import time
from getpass import getpass

import pandas as pd
from dotenv import load_dotenv
from videodb import connect
from videodb.exceptions import InvalidRequestError

load_dotenv()
api_key = os.getenv("VIDEO_DB_API_KEY") or getpass("Enter your VideoDB API key: ")
base_url = os.getenv("VIDEO_DB_BASE_URL") or input("VideoDB endpoint (VIDEO_DB_BASE_URL): ")
conn = connect(api_key=api_key, base_url=base_url)


def show(response):
    print(json.dumps(response, indent=2))

<a id="dataset"></a>
## 2. The dataset

In the Lab, click **Connect** and paste its cell over this one. `SAS_SECRET_NAME` is the name of
a Key Vault secret in your VideoDB deployment that holds a read-only SAS for the container. The SAS
itself never appears here.

In [ ]:
ACCOUNT_URL = "https://<storage-account>.blob.core.windows.net"
CONTAINER = "<container>"
ROOT = "<folder>"
SAS_SECRET_NAME = "<secret-name>"

<a id="connect"></a>
## 3. Connect

Connecting validates the dataset, so connect while it is clean. The cell lists the datasets already
connected and reuses the one that points at this folder, so re-running it never connects twice.

In [ ]:
connected = conn.get(path="dataset")["datasets"]
display(pd.DataFrame(connected, columns=["id", "name", "layout", "location", "created_at"]))

location = f"{ACCOUNT_URL}/{CONTAINER}/{ROOT}".rstrip("/")
DATASET_ID = next((d["id"] for d in connected if d["location"].rstrip("/") == location), None)

In [ ]:
if DATASET_ID:
    print("Already connected:", DATASET_ID)
else:
    try:
        dataset = conn.post(path="dataset", data={
            "name": ROOT,
            "source": {
                "url": f"{ACCOUNT_URL}/{CONTAINER}",
                "root": ROOT,
                "credential": {"kind": "key_vault_secret", "name": SAS_SECRET_NAME},
            },
            "format": {"layout": "lerobot"},
        })
        DATASET_ID = dataset["id"]
        show(dataset)
    except InvalidRequestError as error:
        show(error.response.json())

<a id="validate"></a>
## 4. Validate

Checks the layout: the manifest, the episode index, and that every file they name is there.

In [ ]:
try:
    show(conn.post(path=f"dataset/{DATASET_ID}/validate"))
except InvalidRequestError as error:
    show(error.response.json())

<a id="audit"></a>
## 5. Audit

Decodes every episode's video and runs the `completeness` and `camera_validity` checks. Set
`EPISODES` to a list, such as `[0, 2]`, to audit only those episodes.

In [ ]:
EPISODES = "all"

job = conn.post(path=f"dataset/{DATASET_ID}/audit",
                data={"episodes": EPISODES, "checks": ["completeness", "camera_validity"]})
while (result := conn.session.get(job["output_url"], timeout=30).json())["status"] in ("processing", "in progress"):
    time.sleep(5)
audit = result["response"]
show({k: v for k, v in audit.get("data", audit).items() if k != "episodes"})

In [ ]:
episodes = conn.get(path=f"dataset/{DATASET_ID}/audit/{job['id']}/episodes", params={"limit": 500})["episodes"]
display(pd.DataFrame([
    {"episode": e["episode"], "status": e["status"], **{check: r["status"] for check, r in e["checks"].items()}}
    for e in episodes
]).set_index("episode"))

for e in episodes:
    if e["status"] != "passed":
        print(f"--- episode {e['episode']}")
        show(e["checks"])

<a id="disconnect"></a>
## 6. Disconnect (optional)

Forgets the connection. Curation never changes a connected dataset's files.

In [ ]:
# conn.delete(path=f"dataset/{DATASET_ID}")